# CodSoft Task 2 — Movie Rating Prediction with Python

This project builds a regression model to predict a movie's rating using the uploaded **IMDb Movies India** dataset.

### Workflow
1. Load and inspect the dataset
2. Clean and preprocess the data
3. Perform exploratory data analysis
4. Engineer useful features
5. Prepare categorical and numerical variables
6. Split the data into training and testing sets
7. Train regression models
8. Evaluate the models using MAE, RMSE and R²
9. Compare the models
10. Generate example rating predictions

**Target:** `Rating`

**Note:** The dataset contains movie information such as genre, director, actors, year, duration, votes and rating. The model uses the available columns after preprocessing.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


In [ ]:
# Load the dataset
from pathlib import Path

candidate_paths = [
    Path("IMDb Movies India.csv"),                 # GitHub / Colab / same folder
    Path("/mnt/data/IMDb Movies India.csv")        # ChatGPT runtime
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "IMDb Movies India.csv was not found. Place the CSV in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH, encoding="latin1")

# Clean column names
df.columns = [col.strip() for col in df.columns]

print("Dataset path:", DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# Dataset information
print("Columns:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
# Descriptive statistics
display(df.describe(include="all").T)


## Data Cleaning and Preprocessing

In [ ]:
# Work on a copy
data = df.copy()

# Convert numeric-looking columns safely
data["Year"] = data["Year"].astype(str).str.extract(r"(\d{4})")[0]
data["Year"] = pd.to_numeric(data["Year"], errors="coerce")

data["Duration"] = pd.to_numeric(
    data["Duration"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

data["Votes"] = pd.to_numeric(
    data["Votes"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

data["Rating"] = pd.to_numeric(data["Rating"], errors="coerce")

print("Numeric columns converted successfully.")
display(data[["Year", "Duration", "Votes", "Rating"]].head())


In [ ]:
# Remove rows where the target rating is missing
data = data.dropna(subset=["Rating"]).copy()

print("Rows after removing missing target ratings:", len(data))
print("Missing ratings:", data["Rating"].isnull().sum())


## Exploratory Data Analysis

In [ ]:
# Distribution of movie ratings
plt.figure(figsize=(8, 5))
sns.histplot(data["Rating"], bins=20, kde=True)
plt.title("Distribution of Movie Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Movies")
plt.show()


In [ ]:
# Number of movies by year
year_counts = data["Year"].value_counts().sort_index()

plt.figure(figsize=(12, 5))
plt.plot(year_counts.index, year_counts.values)
plt.title("Number of Movies by Release Year")
plt.xlabel("Year")
plt.ylabel("Number of Movies")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Rating versus number of votes
plt.figure(figsize=(8, 5))
sns.scatterplot(data=data, x="Votes", y="Rating", alpha=0.5)
plt.title("Movie Rating vs Number of Votes")
plt.xlabel("Votes")
plt.ylabel("Rating")
plt.show()


In [ ]:
# Average rating by genre
genre_rating = (
    data.dropna(subset=["Genre"])
        .assign(Genre=data["Genre"].str.split(","))
        .explode("Genre")
        .assign(Genre=lambda x: x["Genre"].str.strip())
        .groupby("Genre")["Rating"]
        .agg(["mean", "count"])
        .query("count >= 20")
        .sort_values("mean", ascending=False)
        .head(15)
)

display(genre_rating)

plt.figure(figsize=(10, 6))
sns.barplot(x=genre_rating["mean"], y=genre_rating.index)
plt.title("Top Genres by Average Rating (minimum 20 movies)")
plt.xlabel("Average Rating")
plt.ylabel("Genre")
plt.show()


In [ ]:
# Correlation between numerical features
numeric_cols = ["Year", "Duration", "Votes", "Rating"]
corr = data[numeric_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


## Feature Engineering

In [ ]:
# Create a simplified feature set.
# We retain Director and the first listed actor/actress because the original
# dataset can contain many unique names, which can make a beginner regression
# model overly sparse.

model_data = data.copy()

# Extract the first actor/actress listed in each column
for col in ["Actor 1", "Actor 2", "Actor 3"]:
    model_data[col] = model_data[col].fillna("Unknown").astype(str).str.strip()

# Combine actor columns into one compact feature
model_data["Lead_Actor"] = model_data["Actor 1"]

# Keep only useful modeling columns
features = [
    "Year", "Duration", "Votes",
    "Genre", "Director", "Lead_Actor"
]

target = "Rating"

model_data = model_data[features + [target]].copy()

display(model_data.head())


In [ ]:
# Define X and y
X = model_data[features]
y = model_data[target]

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)


In [ ]:
# Separate numerical and categorical features
numeric_features = ["Year", "Duration", "Votes"]
categorical_features = ["Genre", "Director", "Lead_Actor"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## Model 1 — Linear Regression

In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

linear_model.fit(X_train, y_train)

pred_lr = linear_model.predict(X_test)

mae_lr = mean_absolute_error(y_test, pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
r2_lr = r2_score(y_test, pred_lr)

print("Linear Regression")
print("MAE :", round(mae_lr, 4))
print("RMSE:", round(rmse_lr, 4))
print("R²  :", round(r2_lr, 4))


## Model 2 — Random Forest Regression

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

rf_model.fit(X_train, y_train)

pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
r2_rf = r2_score(y_test, pred_rf)

print("Random Forest Regression")
print("MAE :", round(mae_rf, 4))
print("RMSE:", round(rmse_rf, 4))
print("R²  :", round(r2_rf, 4))


## Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae_lr, mae_rf],
    "RMSE": [rmse_lr, rmse_rf],
    "R²": [r2_lr, r2_rf]
})

display(results.round(4))


In [ ]:
# Visual comparison of actual and predicted ratings
best_predictions = pred_rf if r2_rf >= r2_lr else pred_lr
best_model_name = "Random Forest" if r2_rf >= r2_lr else "Linear Regression"

plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=best_predictions, alpha=0.5)

min_value = min(y_test.min(), best_predictions.min())
max_value = max(y_test.max(), best_predictions.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.title(f"Actual vs Predicted Ratings — {best_model_name}")
plt.xlabel("Actual Rating")
plt.ylabel("Predicted Rating")
plt.show()


## Example Predictions

In [ ]:
# Show predictions for a few movies from the test set
sample = X_test.head(10).copy()

if best_model_name == "Random Forest":
    sample_predictions = rf_model.predict(sample)
else:
    sample_predictions = linear_model.predict(sample)

prediction_output = sample.copy()
prediction_output["Actual_Rating"] = y_test.loc[sample.index].values
prediction_output["Predicted_Rating"] = np.round(sample_predictions, 2)

display(prediction_output)


## Conclusion

The IMDb Movies India dataset was cleaned and prepared for a regression problem where the target variable is the movie's `Rating`.

Numerical features such as year, duration and votes were converted to usable numeric values. Categorical information such as genre, director and lead actor was encoded using one-hot encoding. Missing feature values were handled using an imputation pipeline.

Linear Regression and Random Forest Regression were trained and evaluated using **MAE, RMSE and R²**. The model with the better R² score was selected as the final model.

The project demonstrates data cleaning, exploratory data analysis, feature engineering, preprocessing, regression modeling and model evaluation required for CodSoft Task 2.
